# 📄 ML Paper Tutor Agent — Google Colab Setup

This notebook sets up the ML Paper Tutor Agent in Google Colab with:
- All dependencies installed
- Ollama running with Qwen2.5-1.5B
- Streamlit UI accessible via ngrok tunnel

**Runtime:** CPU or T4 GPU (free tier works fine)

---

## 1. Install Dependencies

In [ ]:
# Install Python dependencies
!pip install -q langchain langchain-community langchain-huggingface \
    sentence-transformers faiss-cpu PyMuPDF ollama streamlit \
    python-dotenv tiktoken requests pyngrok

## 2. Install & Start Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
import subprocess
import time

process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)  # Wait for server to start
print('Ollama server started!')

In [ ]:
# Pull the Qwen model (~1 GB download)
!ollama pull qwen2.5:1.5b

In [ ]:
# Verify model is available
!ollama list

## 3. Clone / Upload Project

**Option A:** Upload the `paper_agent/` folder to Colab via the file browser.

**Option B:** Clone from Git (if hosted):
```python
!git clone <your-repo-url> paper_agent
```

In [ ]:
# Set working directory
import os
os.chdir('/content/paper_agent')
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

## 4. Test the Pipeline (Without UI)

In [ ]:
# Upload a PDF for testing
from google.colab import files
uploaded = files.upload()  # Upload a PDF via the dialog

# Get the filename
pdf_filename = list(uploaded.keys())[0]
print(f'Uploaded: {pdf_filename}')

In [ ]:
# Ingest the PDF
import sys
sys.path.insert(0, '.')

from ingestion.pipeline import ingest_pdf

result = ingest_pdf(pdf_filename)
print(f"Chunks: {result['stats']['total_chunks']}")
print(f"Pages:  {result['metadata']['pages']}")
print(f"\nFirst chunk preview:")
print(result['chunks'][0].page_content[:200])

In [ ]:
# Create FAISS index
from embeddings.embedder import get_embedding_model
from vectorstore.faiss_store import create_index, get_index_stats

embedding_model = get_embedding_model()
index = create_index(result['chunks'], embedding_model=embedding_model)
print(f"Index stats: {get_index_stats(index)}")

In [ ]:
# Query the paper
from generation.chain import PaperTutorChain
from memory.chat_memory import ChatMemory

chain = PaperTutorChain(index=index, memory=ChatMemory())

# Ask a question
response = chain.ask("What is the main contribution of this paper?")
print(f"\n🤖 Answer:\n{response.answer}")
print(f"\n📎 Sources:")
for src in response.sources:
    print(f"   {src['source']} · Page {src['page']} · {int(src['score']*100)}%")

## 5. Launch Streamlit UI (via ngrok tunnel)

Get a free ngrok auth token at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# Set your ngrok auth token
NGROK_AUTH_TOKEN = 'YOUR_TOKEN_HERE'  # Replace with your token

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Streamlit in background
process = subprocess.Popen(
    ['streamlit', 'run', 'ui/app.py', '--server.port', '8501',
     '--server.headless', 'true'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print(f'\n🌐 Paper Tutor is live at: {public_url}\n')

## 6. Run Tests

In [ ]:
# Run the full test suite
!python -m pytest tests/ -v --tb=short

---

## 💡 Tips

- **GPU acceleration**: If using a T4 runtime, embeddings will automatically use CUDA.
- **Multiple papers**: Upload more PDFs — they're added incrementally to the same index.
- **Memory**: Colab free tier has ~12 GB RAM. The model + embeddings use ~4 GB, leaving plenty.
- **Persistence**: The FAISS index is saved to disk. If the runtime disconnects, re-run cells 1-3 and the index will still be there.